In [22]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import re
pd.options.mode.chained_assignment = None

In [23]:

#! Load data from file
df = pd.read_csv('raw_data/propertyfinder_listings.csv')
#* Grab the numeric columns
numeric_cols = ['Price', 'Number Of Bedrooms', 'Number Of Bathrooms', 'Area']
#* Grab categorical columns
categorical_cols = df.drop(numeric_cols,axis=1).columns
#* total number of rows
data_count = df.count().iloc[0]
#* 1% threshold that will be used to cut off low frequency classes
threshold = data_count // 100
df.head()

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,"5,000,000 SAR",Rest House,4,4,"1,420 sqm","شاطئ نصف القمر, Ad Dammam, Eastern"
1,"6,000,000 SAR",Villa,7+,7+,500 sqm,"Al Wadi, Riyadh, Ar Riyadh"
2,"1,650,000 SAR",Apartment,3,4,144 sqm,"An Narjis, Riyadh, Ar Riyadh"
3,"2,400,000 SAR",Whole Building,7+,7+,423 sqm,"Al Faisaliyah, Jeddah, Makkah Al Mukarramah"
4,"2,500,000 SAR",Villa,7+,2,395 sqm,"Al Muhammadiyah, Jeddah, Makkah Al Mukarramah"


In [24]:
df2 = df.drop_duplicates()

In [25]:

#? function that is used filter non numeric characters in numeric features if value is completely non numeric replace it with NAN
def clean_numeric(x):
    digits = re.sub(r'\D', '', str(x))
    return digits if digits else np.nan


In [ ]:

#? function for cleaning the housing prices data in dataframe
def data_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    #* drop rows containing empty values
    df.dropna(inplace=True)
    #* standardize categorical to be in upper case
    df = df.map(lambda x: str(x).upper())
    #* replace "FULL FLOOR" with "FLOOR" to standarize class names between different sources
    df['Property Type'].map(lambda x: x if x != 'FULL FLOOR' else 'FLOOR')
    #* filter non english characters from "property type" column
    df['Property Type'] = df['Property Type'].astype(str).map(lambda x: re.sub(r'[^A-Za-z\s]', '', x))

    #* clean non-numeric characters from the numeric columns and then casting their type to int
    df[numeric_cols] = df[numeric_cols].map(clean_numeric).astype('Int64')
    #* drop rows containing empty values which is caused by the clean_numeric function
    df.dropna(inplace=True)
    #* filter non english characters from "Location" column while still retaining punctuation characters
    df['Location'] = df['Location'].astype(str).map(lambda x: re.sub(r'[^A-Za-z,.\s]', '', x))
    #* remove property types that have frequency less than the threshold
    df = df.groupby('Property Type').filter(lambda x : len(x)> threshold)
    #* group Locations that have frequency less than the threshold to one value called OTHER
    location_counts = df['Location'].value_counts()
    frequent_locations = location_counts[location_counts >= threshold].index
    df['Location'] = df['Location'].apply(lambda x: x if x in frequent_locations else 'OTHER')
    #* remove white spaces and commas caused by removing arabic characters
    df['Location'] = df['Location'].apply(lambda x: str(x).replace(",      , ", ""))
    #* reset index 
    df = df.reset_index().drop(columns='index',axis=1)

    return df

In [27]:
data_duplicated = data_cleaning(df)
data_nonDuplicated = data_cleaning(df2)
#! Save clean data to csv file
data_nonDuplicated.to_csv('propertyfinder_data/propertyfinder_cleaned_nonDuplicated.csv',index=False)

data_duplicated.to_csv('propertyfinder_data/propertyfinder_cleaned_duplicated.csv',index=False)





